# Demo 2 Report: JAX ViT Inference and Cloud TPU Workflow

> Course: Numerical Computation with JAX  
> Report snapshot: 2026-06-05  
> Repository snapshot: current working tree on 2026-06-05  
> Main topic: pretrained ViT inference, Cloud TPU execution, classifier-head fine-tuning, and checkpoint/resume evidence

This report describes the current Demo 2 workflow in the project. The demo uses `google/vit-base-patch16-224` as a practical JAX/Flax workload. I first run the inference workflow on local CPU, then run the same type of workload on Cloud TPU, retrieve the generated artifacts, and compare the results in report tables. The current branch also adds a small classifier-head fine-tuning workflow that uses the same pretrained ViT backbone but trains only the classifier head.

The earlier pre-TPU report defined the intended path as a local-to-TPU workflow: local CPU baseline, TPU VM setup, JAX TPU backend verification, TPU benchmark run, artifact retrieval, monitoring or observability notes, cleanup, and CPU-vs-TPU comparison. This notebook updates that earlier plan with the evidence now available on the current branch.

The report focuses on evidence that can be checked from code, commands, JSON artifacts, Markdown result tables, or reduced checkpoint/resume summaries. The inference results are timing and workflow evidence. They are not used as a full model-quality evaluation.


## 1. Project objective and report scope

The main goal of the project is to show how a JAX program can move from a local development environment to Google Cloud TPU. Demo 2 is the main presentation path because it uses a real pretrained model while still remaining small enough for a classroom demo.

The report answers three questions:

1. Can the same Demo 2 inference workflow run locally and on Cloud TPU?
2. Can the generated artifacts be retrieved and turned into readable result tables?
3. Can a small training-oriented extension save checkpoints and resume after moving the run state through GCS?

The project now includes local CPU inference, Cloud TPU inference, Imagenette 320 validation-manifest timing, and an optional classifier-head fine-tuning extension. The fine-tuning extension freezes the ViT backbone and trains only the classifier head, so it should be read as a checkpoint/resume workflow demo, not as full ViT fine-tuning.

This report uses three levels of evidence. The first level is application-level evidence, such as JSON metrics, backend name, devices, batch size, image count, timing, throughput, and predictions. The second level is cloud-workflow evidence, such as TPU resource creation, SSH setup, artifact retrieval, GCS copies, and cleanup. The third level is interpretation: what the numbers show, what they do not show, and what limitations remain.


## 2. Baseline before the TPU work

Before the TPU work, `report/progress_report_pre_tpu.md` recorded the local CPU stage of Demo 2. At that time, the project already had a working inference script for `google/vit-base-patch16-224`, JSON outputs, and a comparison helper that could generate Markdown tables for the report.

That baseline was important because it fixed the local workflow before adding cloud complexity. The same script family, result format, and comparison tools could then be reused after the TPU artifacts were retrieved.

The pre-TPU report used Imagenette 320 `val256` as the main CPU evidence because all three batch sizes used 256 real images and no padded images. The local laptop CPU and the newer external CPU machine gave the following baseline:

| Machine / environment | Batch size | Images | Padded | Mean step (s) | Throughput | Within-scope speedup |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| older local laptop CPU | 1 | 256 | 0 | 0.449014 | 2.2271 img/s | 1x |
| older local laptop CPU | 4 | 256 | 0 | 1.51115 | 2.6470 img/s | 1.19x |
| older local laptop CPU | 8 | 256 | 0 | 2.53903 | 3.1508 img/s | 1.41x |
| newer external CPU machine | 1 | 256 | 0 | 0.0834544 | 11.9826 img/s | 1x |
| newer external CPU machine | 4 | 256 | 0 | 0.284359 | 14.0667 img/s | 1.17x |
| newer external CPU machine | 8 | 256 | 0 | 0.580369 | 13.7843 img/s | 1.15x |

The older local CPU improved from about 2.23 images/sec at batch size 1 to about 3.15 images/sec at batch size 8. The newer external CPU was much faster overall, but its batch-size scaling was smaller. This result was useful as a baseline and as a check that the workflow could run on more than one CPU environment.

This was not a controlled hardware benchmark. The two CPU machines had different hardware, operating environments, cache states, and thermal behavior. The point was to establish a reproducible measurement framework before moving the same workflow to Cloud TPU.

The missing part in the pre-TPU report was actual TPU evidence. The current branch adds that evidence through retrieved TPU inference artifacts, Imagenette 320 TPU timing tables, and a small Orbax checkpoint/resume workflow for classifier-head fine-tuning.


## 3. JAX/Flax inference implementation

The main inference script is `examples/pretrained_vit_inference.py`. It uses Hugging Face `AutoImageProcessor` for preprocessing and `FlaxViTForImageClassification` for the pretrained model. The model architecture and pretrained weights are not designed by this project. The project work is the reproducible JAX/Flax execution path around the model.

The script supports both a single image and a manifest of image paths. In manifest mode, the script builds mixed-image batches, pads the final incomplete batch when needed, and records how many padded images were added. Padded entries are not counted as real input images.

The measured path is a jitted inference function. The report does not claim that the speed difference comes only from `jax.jit`, because there is no non-jit ablation here. The important point is that the same jitted path is warmed up first and then measured with synchronization.

```python
@jax.jit
def inference_step(model_params: Mapping[str, Any], inputs: Any) -> Any:
    return model(pixel_values=inputs, params=model_params, train=False).logits

for _ in range(warmup_steps):
    for input_batch in input_batches:
        logits = inference_step(params, input_batch)
        logits.block_until_ready()

for _ in range(benchmark_steps):
    for input_batch in input_batches:
        start_time = time.perf_counter()
        logits = inference_step(params, input_batch)
        logits.block_until_ready()
```

The `block_until_ready()` call is important because JAX may dispatch work asynchronously. Without it, the timer can stop after the work is submitted to the backend, not after the actual inference computation is finished.

Batch size changes the input tensor shape passed to the model. In manifest mode, the code builds fixed-size batches and pads only the final batch when necessary:

```python
batch_specs = build_manifest_batch_specs(
    num_images=len(image_paths),
    batch_size=batch_size,
)
input_batches = [
    pad_manifest_batch(
        pixel_values[batch_spec["start_index"] : batch_spec["end_index"]],
        padding_count=batch_spec["padding_count"],
        jnp_module=jnp,
    )
    for batch_spec in batch_specs
]
```

Each JSON artifact records the selected JAX platform, the actual backend, visible devices, batch size, warmup steps, benchmark steps, number of images, number of batches, padding count, mean step time, total timed inference time, throughput, and per-image top-1 predictions. This makes local CPU artifacts and TPU artifacts comparable through the same report-table workflow.


### Setup

This notebook uses existing artifacts from the repository checkout. It does not rerun benchmarks, download datasets or model weights, create cloud resources, or start TPU jobs. If an expected artifact is not available, the notebook reports the missing path instead of filling in placeholder results.


In [ ]:
from __future__ import annotations

import csv
import json
import math
from pathlib import Path
from typing import Any

try:
    from IPython.display import Markdown, display
except ImportError:  # Allows basic script-style execution outside Jupyter.
    Markdown = None

    def display(value: Any) -> None:
        print(value)

try:
    import matplotlib.image as mpimg
    import matplotlib.pyplot as plt
except ImportError as exc:
    mpimg = None
    plt = None
    print(f"matplotlib is unavailable: {exc}")


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root from the current notebook location."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "README.md").is_file():
            return candidate
    raise FileNotFoundError("Could not find a repository root containing both pyproject.toml and README.md")


REPO_ROOT = find_repo_root()
RESULTS_DIR = REPO_ROOT / "report" / "results"
INFERENCE_RUNS_DIR = REPO_ROOT / "runs" / "vit-inference"
FINETUNE_RUNS_DIR = REPO_ROOT / "runs" / "vit-finetune"

print(f"repo root: {REPO_ROOT}")
print(f"results dir exists: {RESULTS_DIR.exists()}")
print(f"inference runs dir exists: {INFERENCE_RUNS_DIR.exists()}")
print(f"fine-tune runs dir exists: {FINETUNE_RUNS_DIR.exists()}")


## 4. Result artifacts

The report-ready tables are stored under `report/results/`. They summarize selected JSON artifacts, including local CPU results, Cloud TPU results, Imagenette 320 timing results, and the public-example CPU-vs-TPU smoke comparison.

This section shows the Markdown tables directly. Since each table only has a few rows, the table view is clearer than an additional figure.

In [ ]:
def show_markdown_file(relative_path: str, *, max_chars: int | None = None) -> None:
    path = REPO_ROOT / relative_path
    if not path.exists():
        display(Markdown(f"> Missing expected Markdown artifact: `{relative_path}`"))
        return
    text = path.read_text(encoding="utf-8")
    if max_chars is not None and len(text) > max_chars:
        text = text[:max_chars].rstrip() + "\n\n> Output truncated for notebook display."
    display(Markdown(f"### `{relative_path}`\n\n" + text))


REPORT_TABLES = [
    "report/results/demo2_local_private_examples_cpu.md",
    "report/results/demo2_cloud_imagenette320_val64_tpu.md",
    "report/results/demo2_cloud_imagenette320_val256_tpu.md",
    "report/results/demo2_cloud_imagenette320_valfull_tpu.md",
    "report/results/demo2_local_cpu_vs_cloud_tpu_public_examples_b4.md",
    "report/results/demo2_imagenette320_overview.md",
    "report/results/demo2_imagenette320_batch_scaling.md",
    "report/results/demo2_imagenette320_cpu_vs_tpu.md",
]

for table_path in REPORT_TABLES:
    show_markdown_file(table_path)


## 5. Visual inference examples

The visual grids show selected images together with the model's top-1 prediction. They are used only as a qualitative sanity check for saved prediction artifacts.

The `demo2_cloud_imagenette320_val64_tpu_b4` grid is intentionally not shown here. In the current artifact, the selected images come from the same Imagenette class folder, so the grid does not add useful information to the report. The timing table is still kept in the result-artifact section.

The notebook keeps the local private-example grid and, when local images and JSON are available, the full-validation TPU grid. Those grids are more useful for checking that the saved prediction records and image paths are connected correctly.

In [ ]:
def load_prediction_payload(json_path: Path) -> dict[str, Any] | None:
    """Load one inference JSON artifact."""
    if not json_path.exists():
        display(Markdown(f"> Missing expected prediction JSON: `{json_path.relative_to(REPO_ROOT)}`"))
        return None
    try:
        return json.loads(json_path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        display(Markdown(f"> Could not parse `{json_path.relative_to(REPO_ROOT)}`: {exc}"))
        return None


def load_prediction_records(json_path: Path) -> list[dict[str, Any]]:
    """Load per-image prediction records from current or legacy JSON artifacts."""
    payload = load_prediction_payload(json_path)
    if payload is None:
        return []

    records = payload.get("image_results")
    if isinstance(records, list):
        return [dict(record) for record in records]

    predictions = payload.get("predictions")
    if isinstance(predictions, list):
        return [dict(record) for record in predictions]

    if payload.get("image_path") or payload.get("predicted_label"):
        return [dict(payload)]

    display(Markdown(f"> No per-image prediction records found: `{json_path.relative_to(REPO_ROOT)}`."))
    return []


def resolve_image_path(path_like: str | None, repo_root: Path) -> Path | None:
    """Resolve a JSON image path against the repo and likely ignored local image roots."""
    if not path_like:
        return None
    raw = Path(path_like)
    candidates: list[Path] = []
    if raw.is_absolute():
        candidates.append(raw)
    else:
        candidates.extend([repo_root / raw, Path.cwd() / raw])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    filename = raw.name
    if not filename:
        return None
    search_roots = [
        repo_root / "data" / "local" / "demo2_vit_images",
        repo_root / "data" / "local" / "imagenette2-320" / "val",
        repo_root / "data" / "local" / "imagenette2-320" / "train",
        repo_root / "examples" / "assets",
    ]
    for base in search_roots:
        if not base.exists():
            continue
        matches = list(base.rglob(filename))
        if matches:
            return matches[0]
    return None


def record_image_path_text(record: dict[str, Any]) -> str:
    return str(record.get("image_path") or record.get("path") or record.get("filename") or "")


def prediction_label(record: dict[str, Any]) -> str:
    return str(record.get("predicted_label") or record.get("top1_label") or "label unavailable")


def prediction_score(record: dict[str, Any]) -> float | None:
    value = record.get("predicted_score")
    if isinstance(value, int | float) and not isinstance(value, bool):
        return float(value)
    top5 = record.get("top5")
    if isinstance(top5, list) and top5:
        score = top5[0].get("score") if isinstance(top5[0], dict) else None
        if isinstance(score, int | float) and not isinstance(score, bool):
            return float(score)
    return None


def inferred_class_key(record: dict[str, Any]) -> str:
    """Infer a grouping key from explicit labels or from the parent class folder."""
    for key in ("true_label", "ground_truth_label", "class_label", "label_name", "label", "label_id"):
        value = record.get(key)
        if isinstance(value, str) and value.strip():
            return value.strip()

    path_text = record_image_path_text(record)
    if not path_text:
        return "unknown"
    raw = Path(path_text)

    # Imagenette paths usually look like .../val/n01440764/image.JPEG.
    for part in reversed(raw.parts[:-1]):
        if part.startswith("n") and part[1:].isdigit():
            return part

    if len(raw.parts) >= 2:
        return raw.parts[-2]
    return "unknown"


def selected_class_summary(records: list[dict[str, Any]]) -> str:
    counts: dict[str, int] = {}
    for record in records:
        key = inferred_class_key(record)
        counts[key] = counts.get(key, 0) + 1
    parts = [f"{key}: {counts[key]}" for key in sorted(counts)]
    return ", ".join(parts) if parts else "none"


def select_diverse_records(records: list[dict[str, Any]], max_images: int = 12) -> list[dict[str, Any]]:
    """Select records across inferred class folders before filling remaining slots.

    This avoids the common notebook mistake of displaying the first N sorted
    Imagenette records, which can all come from the same class directory.
    """
    if len(records) <= max_images:
        return records

    grouped: dict[str, list[dict[str, Any]]] = {}
    for record in records:
        grouped.setdefault(inferred_class_key(record), []).append(record)

    if len(grouped) <= 1:
        # If the artifact itself only contains one inferred class, spread the
        # display across the file order so the grid is not just the first images.
        step = max(1, math.floor(len(records) / max_images))
        return records[::step][:max_images]

    selected: list[dict[str, Any]] = []
    group_keys = sorted(grouped)
    index = 0
    while len(selected) < max_images:
        added_this_round = False
        for key in group_keys:
            group = grouped[key]
            if index < len(group):
                selected.append(group[index])
                added_this_round = True
                if len(selected) >= max_images:
                    break
        if not added_this_round:
            break
        index += 1
    return selected


def show_prediction_grid(records: list[dict[str, Any]], title: str, max_images: int = 12) -> None:
    if plt is None or mpimg is None:
        print("matplotlib is unavailable; skipping image grid")
        return
    if not records:
        display(Markdown(f"> No displayable records for `{title}`."))
        return

    shown = select_diverse_records(records, max_images=max_images)
    class_count = len({inferred_class_key(record) for record in shown})
    display(
        Markdown(
            f"**{title}**: showing {len(shown)} images from {class_count} "
            f"inferred class folder(s). Selected class counts: {selected_class_summary(shown)}."
        )
    )

    cols = min(4, len(shown))
    rows = math.ceil(len(shown) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(3.1 * cols, 3.8 * rows))
    axes_list = list(getattr(axes, "flat", [axes]))
    fig.suptitle(title, fontsize=13)

    for axis, record in zip(axes_list, shown, strict=False):
        image_path = resolve_image_path(record_image_path_text(record), REPO_ROOT)
        axis.axis("off")
        if image_path is None or not image_path.exists():
            axis.text(0.5, 0.5, "missing image", ha="center", va="center")
            filename = Path(record_image_path_text(record) or "unknown").name
        else:
            try:
                axis.imshow(mpimg.imread(image_path))
            except Exception as exc:  # keep report notebook robust for corrupt/missing local files
                axis.text(0.5, 0.5, f"unreadable image\n{exc}", ha="center", va="center")
            filename = image_path.name

        label = prediction_label(record)
        score = prediction_score(record)
        class_key = inferred_class_key(record)
        score_text = f"\nscore={score:.3f}" if score is not None else ""
        axis.set_title(f"{filename}\nfolder={class_key}\npred={label}{score_text}", fontsize=8)

    for axis in axes_list[len(shown):]:
        axis.axis("off")
    plt.tight_layout()
    plt.show()


GRID_SPECS = [
    (
        INFERENCE_RUNS_DIR / "demo2_local_private_examples_cpu_b4.json",
        "demo2_local_private_examples_cpu_b4",
    ),
    (
        INFERENCE_RUNS_DIR / "demo2_cloud_imagenette320_valfull_tpu_b4.json",
        "demo2_cloud_imagenette320_valfull_tpu_b4_diverse_visual_check",
    ),
]

for json_path, title in GRID_SPECS:
    records = load_prediction_records(json_path)
    show_prediction_grid(records, title, max_images=12)


## 6. Cloud TPU execution path

The TPU part has two purposes. First, it shows that the pretrained ViT inference script can run on a TPU backend. Second, it shows that the generated JSON artifacts can be copied back and summarized by the same local report tools.

This section follows the same local-to-TPU plan described in the earlier pre-TPU report: keep a local CPU baseline, create or wait for a TPU VM, verify the JAX TPU backend, run the same benchmark family, retrieve the JSON artifacts, generate local comparison tables, and clean up cloud resources.

The full workflow is documented in `cloud/demo2_tpu_quickstart.md`. The commands below are kept as classroom demonstration references, while the report interpretation focuses on the resulting artifacts and evidence.

The main execution path uses a queued resource. In the completed runs, the successful public-example smoke path used a `v6e-1` spot TPU in `us-east1-d`. The fine-tuning resume run later used a `v6e-1` spot TPU in `europe-west4-a`. These zones and shapes are evidence from this project, not a guarantee that the same capacity will always be available.

### Local Ubuntu / WSL repo root: local CPU baseline

```bash
uv sync --frozen --group pretrained

uv run --group pretrained python examples/pretrained_vit_inference.py   --jax-platform cpu   --image-manifest examples/assets/manifest.txt   --batch-size 4   --warmup-steps 1   --benchmark-steps 5   --output runs/vit-inference/demo2_local_public_examples_cpu_b4.json
```

### Google Cloud Shell or local gcloud terminal: resource variables

```bash
export PROJECT_ID="<PROJECT_ID>"
export REGION="us-east1"
export ZONE="us-east1-d"
export TPU_NAME="demo2-vit-v6e1-use1-spot"
export QUEUED_RESOURCE_ID="${TPU_NAME}-qr"
export ACCELERATOR_TYPE="v6e-1"
export RUNTIME_VERSION="v2-alpha-tpuv6e"
export NETWORK_NAME="default"
export SUBNET_NAME="default"
```

### Google Cloud Shell or local gcloud terminal: preflight checks

```bash
gcloud auth login
gcloud config set project "$PROJECT_ID"
gcloud config set compute/zone "$ZONE"
gcloud services enable tpu.googleapis.com

gcloud compute networks list --project "$PROJECT_ID"
gcloud compute networks subnets list --project "$PROJECT_ID" --regions "$REGION"
gcloud compute networks subnets describe "$SUBNET_NAME" --project "$PROJECT_ID" --region "$REGION"

gcloud compute tpus accelerator-types list --project "$PROJECT_ID" --zone "$ZONE"
gcloud compute tpus versions list --project "$PROJECT_ID" --zone "$ZONE"
gcloud compute tpus queued-resources list --project "$PROJECT_ID" --zone "$ZONE"
gcloud compute tpus tpu-vm list --project "$PROJECT_ID" --zone "$ZONE"
```

### Google Cloud Shell or local gcloud terminal: create and wait for a spot queued resource

```bash
gcloud compute tpus queued-resources create "$QUEUED_RESOURCE_ID"   --node-id "$TPU_NAME"   --project "$PROJECT_ID"   --zone "$ZONE"   --accelerator-type "$ACCELERATOR_TYPE"   --runtime-version "$RUNTIME_VERSION"   --network "$NETWORK_NAME"   --subnetwork "$SUBNET_NAME"   --spot
```

```bash
while true; do
  date
  gcloud compute tpus queued-resources describe "$QUEUED_RESOURCE_ID"     --project "$PROJECT_ID"     --zone "$ZONE"     --format="value(state)"
  sleep 20
done
```

`ACTIVE` is the state needed before SSH. If the resource stays in `WAITING_FOR_RESOURCES` for too long, or if the TPU VM becomes unusable because of maintenance or preemption, the resource should be deleted before trying another zone or shape.

### TPU VM shell: checkout and backend verification

Use the branch, tag, or commit that contains the Demo 2 TPU workflow. For a reproducible rerun, record the resolved commit with `git rev-parse HEAD` or replace `COMMIT_SHA` with a real commit SHA.

```bash
export REPO_URL="https://github.com/anthroplankton/numerical-jax-project.git"
export BRANCH="<BRANCH_OR_TAG_WITH_DEMO2_TPU_WORKFLOW>"
# export COMMIT_SHA="<real commit SHA>"  # optional reproducible mode

git clone "$REPO_URL" numerical-jax-project
cd numerical-jax-project
git fetch --all --prune
git switch "$BRANCH"
# git checkout "$COMMIT_SHA"  # only after replacing the placeholder

git status --short --branch
git rev-parse HEAD

curl -LsSf https://astral.sh/uv/install.sh | sh
export PATH="$HOME/.local/bin:$PATH"
uv --version

uv sync --frozen --group pretrained
uv pip install -U "jax[tpu]"   -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

uv run python -c "import jax; print('jax_version=', jax.__version__); print('default_backend=', jax.default_backend()); print('device_count=', jax.device_count()); print('local_device_count=', jax.local_device_count()); print('devices=', jax.devices()); raise SystemExit(0 if jax.default_backend() == 'tpu' else 1)"
```

The backend check is part of the evidence. A run should not be called TPU evidence unless `jax.default_backend()` reports `tpu` and TPU devices are visible.

### TPU VM shell: public-example inference smoke run

```bash
uv run --group pretrained python examples/pretrained_vit_inference.py   --jax-platform tpu   --image-manifest examples/assets/manifest.txt   --batch-size 4   --warmup-steps 1   --benchmark-steps 5   --output runs/vit-inference/demo2_cloud_public_examples_tpu_b4.json
```

### TPU VM shell: Imagenette 320 inference runs

The Imagenette path must exist on the TPU VM before running these commands:

```text
data/local/imagenette2-320/val
```

The retrieved artifact families use `manifest_val_64.txt`, `manifest_val_256.txt`, and `manifest_val_full.txt` with batch sizes `1`, `4`, and `8`. The `val64` `b4` command is:

```bash
uv run --group pretrained python examples/pretrained_vit_inference.py   --jax-platform tpu   --image-manifest data/local/imagenette2-320/val/manifest_val_64.txt   --batch-size 4   --warmup-steps 1   --benchmark-steps 5   --output runs/vit-inference/demo2_cloud_imagenette320_val64_tpu_b4.json
```

The same pattern is used for `val256` and `val_full`, and for batch sizes `1` and `8`.

### Google Cloud Shell or local gcloud terminal: retrieve artifacts

```bash
mkdir -p runs

gcloud compute tpus tpu-vm scp --recurse   "$TPU_NAME":~/numerical-jax-project/runs/vit-inference   runs/   --project "$PROJECT_ID"   --zone "$ZONE"
```

### Local Ubuntu / WSL repo root: generate comparison tables

```bash
uv run python scripts/compare_vit_results.py   runs/vit-inference/demo2_local_public_examples_cpu_b4.json   runs/vit-inference/demo2_cloud_public_examples_tpu_b4.json   --output runs/vit-inference/demo2_local_cpu_vs_cloud_tpu_public_examples_b4_compare.json   --markdown-output report/results/demo2_local_cpu_vs_cloud_tpu_public_examples_b4.md

uv run python scripts/generate_vit_summary_tables.py   --input-dir runs/vit-inference   --output-dir report/results
```

### Google Cloud Shell or local gcloud terminal: cleanup

```bash
gcloud compute tpus queued-resources delete "$QUEUED_RESOURCE_ID"   --project "$PROJECT_ID"   --zone "$ZONE"   --force

gcloud compute tpus queued-resources list --project "$PROJECT_ID" --zone "$ZONE"
gcloud compute tpus tpu-vm list --project "$PROJECT_ID" --zone "$ZONE"
```


## 7. Optional extension: classifier-head fine-tuning

The optional fine-tuning part keeps the same pretrained ViT model, but changes the task from inference to a small training-oriented workflow. The ViT backbone is frozen. Only the classifier head is trained.

For this snapshot, Demo 2 is not only an inference demo: the main benchmark evidence is inference timing, and the optional extension adds classifier-head training workflow evidence.

This design keeps the run small enough for a course demo and makes the checkpoint state easy to inspect. The goal is not to train a high-quality Imagenette model. The goal is to show that a TPU training workflow can save state, copy completed checkpoints to durable storage, and resume later.

### TPU VM shell: dependency setup

```bash
uv sync --frozen --group pretrained --group training

uv pip install -U "jax[tpu]" \
  -f https://storage.googleapis.com/jax-releases/libtpu_releases.html

uv run python -c "import jax; print('default_backend=', jax.default_backend()); print('devices=', jax.devices()); raise SystemExit(0 if jax.default_backend() == 'tpu' else 1)"
```

### TPU VM shell: class-balanced smoke manifests

The fine-tuning smoke profiles use small balanced manifests. This avoids the visual and metric skew that can happen when a sorted Imagenette manifest is truncated by a simple global limit.

```bash
uv run python scripts/build_image_manifest.py \
  data/local/imagenette2-320/train \
  --output data/local/imagenette2-320/train/manifest_train_balanced_50.txt \
  --per-class-limit 5

uv run python scripts/build_image_manifest.py \
  data/local/imagenette2-320/val \
  --output data/local/imagenette2-320/val/manifest_val_balanced_50.txt \
  --per-class-limit 5
```


With `--per-class-limit 5`, the train manifest has 50 images: 5 images from each of the 10 Imagenette class folders. With batch size 8, the training batches have this real-example pattern:

```text
8, 8, 8, 8, 8, 8, 2
```

The last batch is still padded to the fixed JAX shape for batch size 8, but it contains only 2 real examples. This detail matters when reading the `examples_per_second` plot later.

### TPU VM shell: short learning-curve smoke profile

```bash
export RUN_NAME="demo2_cloud_vit_head_finetune_tpu_curve"
export RUN_DIR="$(pwd)/runs/vit-finetune/$RUN_NAME"
export CKPT_DIR="$RUN_DIR/checkpoints"
mkdir -p "$RUN_DIR" "$CKPT_DIR"

uv run --group pretrained --group training python examples/demo2_pretrained_vit_finetune.py \
  --jax-platform tpu \
  --train-manifest data/local/imagenette2-320/train/manifest_train_balanced_50.txt \
  --eval-manifest data/local/imagenette2-320/val/manifest_val_balanced_50.txt \
  --batch-size 8 \
  --learning-rate 0.001 \
  --max-steps 300 \
  --min-train-seconds 0 \
  --checkpoint-every-steps 100 \
  --checkpoint-every-seconds 0 \
  --eval-every-steps 25 \
  --checkpoint-dir "$CKPT_DIR" \
  --output-dir "$RUN_DIR" \
  --save-predictions
```

### TPU VM shell and GCS: checkpoint / resume evidence profile

Completed artifacts and Orbax checkpoints can be copied to GCS after a run segment:

```bash
export RUN_NAME="demo2_cloud_vit_head_finetune_tpu_resume_first"
export RUN_DIR="$(pwd)/runs/vit-finetune/$RUN_NAME"
export CKPT_DIR="$RUN_DIR/checkpoints"
export GCS_RUN_ROOT="gs://$BUCKET_NAME/numerical-jax-project/demo2-vit-finetune"
mkdir -p "$RUN_DIR" "$CKPT_DIR"

uv run --group pretrained --group training python examples/demo2_pretrained_vit_finetune.py \
  --jax-platform tpu \
  --train-manifest data/local/imagenette2-320/train/manifest_train_balanced_50.txt \
  --eval-manifest data/local/imagenette2-320/val/manifest_val_balanced_50.txt \
  --batch-size 8 \
  --learning-rate 0.001 \
  --max-steps 300 \
  --min-train-seconds 0 \
  --checkpoint-every-steps 100 \
  --checkpoint-every-seconds 0 \
  --eval-every-steps 50 \
  --checkpoint-dir "$CKPT_DIR" \
  --output-dir "$RUN_DIR" \
  --save-predictions

export GCS_RUN_URI="$GCS_RUN_ROOT/artifacts/$RUN_NAME"
export GCS_CKPT_URI="$GCS_RUN_ROOT/checkpoints/$RUN_NAME"
gcloud storage rsync --recursive "$RUN_DIR" "$GCS_RUN_URI"
gcloud storage rsync --recursive "$CKPT_DIR" "$GCS_CKPT_URI"
gcloud storage ls "$GCS_CKPT_URI/"
```

A later TPU VM can restore the checkpoint directory from GCS and continue with `--resume`:

```bash
export BASE_RUN_NAME="demo2_cloud_vit_head_finetune_tpu_resume_first"
export RESUME_RUN_NAME="demo2_cloud_vit_head_finetune_tpu_resume"
export RESUME_RUN_DIR="$(pwd)/runs/vit-finetune/$RESUME_RUN_NAME"
export RESUME_CKPT_DIR="$RESUME_RUN_DIR/checkpoints"
export GCS_CKPT_URI="$GCS_RUN_ROOT/checkpoints/$BASE_RUN_NAME"

mkdir -p "$RESUME_RUN_DIR" "$RESUME_CKPT_DIR"
gcloud storage rsync --recursive "$GCS_CKPT_URI" "$RESUME_CKPT_DIR"
find "$RESUME_CKPT_DIR" -name "*.orbax-checkpoint-tmp" -type d -prune -exec rm -rf {} +

uv run --group pretrained --group training python examples/demo2_pretrained_vit_finetune.py \
  --jax-platform tpu \
  --train-manifest data/local/imagenette2-320/train/manifest_train_balanced_50.txt \
  --eval-manifest data/local/imagenette2-320/val/manifest_val_balanced_50.txt \
  --batch-size 8 \
  --learning-rate 0.001 \
  --max-steps 500 \
  --min-train-seconds 0 \
  --checkpoint-every-steps 100 \
  --checkpoint-every-seconds 0 \
  --eval-every-steps 50 \
  --checkpoint-dir "$RESUME_CKPT_DIR" \
  --output-dir "$RESUME_RUN_DIR" \
  --resume \
  --save-predictions
```


## 8. Orbax checkpoint and resume design

The Orbax part follows the same reporting style as the JAX inference section. The point is not only that the code calls a checkpoint library. The important design choice is what state is saved, when it is saved, how the saved state is checked, and how the run continues after restore.

The fine-tuning extension freezes the ViT backbone and trains only the classifier head. Therefore the checkpoint state is intentionally small. It contains the trainable head parameters, optimizer state, current step, and metadata. The frozen backbone and the Hugging Face model cache are not checkpointed.

```python
@dataclass(frozen=True)
class HeadTrainState:
    head_params: Any
    optimizer_state: Any
    step: int
```

The checkpoint metadata records the identity of the run and the settings needed for a compatibility check. This is similar to the inference JSON metadata: it keeps the result connected to the model, manifests, batch size, learning rate, seed, label set, and Git metadata.

```python
def build_checkpoint_metadata(...):
    return {
        "mode": MODE,
        "model_name": model_name,
        "trainable_scope": TRAINABLE_SCOPE,
        "frozen_scope": FROZEN_SCOPE,
        "train_manifest": str(train_manifest),
        "eval_manifest": str(eval_manifest),
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "eval_every_steps": eval_every_steps,
        "reinit_head": reinit_head,
        "seed": seed,
        "label_ids": list(label_ids),
        **dict(git_metadata),
    }
```

The actual Orbax checkpoint item is limited to the allowed state boundary:

```python
def build_checkpoint_item(*, head_params, optimizer_state, step, metadata):
    return {
        "head_params": head_params,
        "optimizer_state": optimizer_state,
        "step": np.asarray(step, dtype=np.int64),
        "metadata_json": encode_checkpoint_metadata(metadata),
    }

def checkpoint_contains_allowed_keys(item):
    return set(item) == {"head_params", "optimizer_state", "step", "metadata_json"}
```

Saving uses an Orbax `CheckpointManager`. The code waits for async checkpoint writes to finish before treating the checkpoint as complete:

```python
def save_training_checkpoint(manager, *, step, head_params, optimizer_state, metadata):
    from orbax import checkpoint as ocp

    item = build_checkpoint_item(
        head_params=head_params,
        optimizer_state=optimizer_state,
        step=step,
        metadata=metadata,
    )
    manager.save(step, args=ocp.args.StandardSave(item))
    manager.wait_until_finished()
    return step
```

When resuming, the script restores the latest available checkpoint into a target PyTree. It then rebuilds the training state from the restored head parameters, optimizer state, and step number.

```python
restored = restore_latest_checkpoint(checkpoint_manager, target=target)
if restored is not None:
    state = HeadTrainState(
        head_params=restored["head_params"],
        optimizer_state=restored["optimizer_state"],
        step=int(np.asarray(restored["step"])),
    )
    restored_metadata = decode_checkpoint_metadata(restored["metadata_json"])
    validate_restored_checkpoint_metadata(restored_metadata, checkpoint_metadata)
    resumed_from_checkpoint = True
```

The metadata validation is important. A checkpoint should not be accepted blindly just because a directory exists. The code checks identity fields such as `mode`, `model_name`, `trainable_scope`, and `frozen_scope` before continuing. This reduces the risk of resuming from a checkpoint that belongs to a different workflow.

In this project, Orbax writes local checkpoint files under the TPU VM run directory first. GCS is used as the durable copy layer after a checkpoint or run segment is complete. This keeps the classroom workflow easy to understand: save locally with Orbax, copy completed checkpoint directories to GCS, restore them on a later TPU VM, then run with `--resume`.

This design is not a full production fault-tolerance system. If a spot TPU is interrupted before the completed checkpoint is copied to GCS, local-only checkpoint files may be lost. For this course report, the important result is that the state boundary, metadata, local checkpointing, GCS copy, restore, and resume steps are all visible and testable.


## 9. GCS copy and resume evidence

The reduced report artifact `report/results/demo2_cloud_vit_head_finetune_tpu_resume_gcs.md` records the observed checkpoint/resume evidence.

| Item | Observed value |
| --- | --- |
| First run TPU shape | `v6e-1` spot, `us-east1-d` |
| Interruption evidence | real spot or maintenance interruption after the first run |
| Durable checkpoint copies | GCS checkpoint steps `15100`, `15120`, `15140` |
| Resume TPU shape | `v6e-1` spot, `europe-west4-a` |
| Resume backend | `tpu` |
| Resumed from checkpoint | `true` |
| Resume start step | `15140` |
| Resume final step | `51538` |
| Trainable scope | `classifier_head_only` |
| Frozen scope | `vit_backbone` |

The important result is operational. The first TPU run produced Orbax checkpoints, completed checkpoint directories were copied to GCS, and a later TPU VM restored the checkpoint and continued from step `15140`.

This evidence has the same three-level structure as the TPU inference workflow. The application-level evidence is the fine-tuning summary: backend, resume flag, start step, final step, trainable scope, and frozen scope. The cloud-workflow evidence is the GCS copy of completed checkpoint directories and the later restore on a replacement TPU VM. The report-level interpretation is that the workflow can recover from a resource change when a completed checkpoint has already been copied to durable storage.

This evidence does not prove model quality. It shows that the small classifier-head training workflow can survive a resource change when completed checkpoints have been copied to durable storage.

A limitation remains: a post-run or segment-end `rsync` only protects checkpoints that were already copied to GCS. If a spot TPU is interrupted before the copy, local-only checkpoint directories may be lost. For a classroom project, segmented runs plus GCS copies are enough to demonstrate the concept, but they are not the same as a production fault-tolerant training system.


## 10. Fine-tuning metrics and throughput interpretation

The next cell reads local fine-tuning artifacts when they exist, such as `summary.json`, `metrics.csv`, `eval_metrics.csv`, and `train.log` under `runs/vit-finetune/`.

The plots are useful for checking that the smoke run produced metrics over time. They should not be interpreted as an Imagenette accuracy study, especially when the run uses tiny balanced manifests.

One important interpretation point is the two-band pattern that can appear in the `examples_per_second` plot. In the observed fine-tuning profile, the train manifest has 50 images and the batch size is 8. This gives six full batches with 8 real examples and one final padded batch with only 2 real examples.

The code cycles through the training batches with the current step number. It times only the `train_step` call and waits with `loss.block_until_ready()` before stopping the timer. The metric row then reports throughput as `batch.real_size / step_time`, not padded batch size divided by step time.

```python
batch = train_batches[state.step % len(train_batches)]
step_start = time.perf_counter()
head_params, optimizer_state, loss, accuracy = train_step(...)
loss.block_until_ready()
step_time = time.perf_counter() - step_start

"examples_per_second": batch.real_size / step_time
```

Therefore the low throughput band is expected when the padded final batch is used. If the step time is similar, the final batch reports about `2 / 8 = 1/4` of the full-batch throughput. This should not be read as the TPU suddenly becoming four times slower. It is mainly a consequence of the throughput definition and the small 50-example manifest.

The same reasoning also helps rule out two common wrong explanations. Checkpoint writes happen after `step_time` is measured, and periodic evaluation is also outside the measured training-step time for that row. The low band is mainly a padded-batch accounting effect, not checkpoint overhead or evaluation overhead.


In [ ]:
def load_json_if_exists(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        display(Markdown(f"> Could not parse `{path.relative_to(REPO_ROOT)}`: {exc}"))
        return None


def load_csv_records(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    with path.open(newline="", encoding="utf-8") as handle:
        return [dict(row) for row in csv.DictReader(handle)]


def to_float(value: Any) -> float | None:
    """Convert a table or CSV value to float when possible."""
    if value is None:
        return None
    text = str(value).strip().replace("x", "")
    if text.lower() in {"", "n/a", "not run", "nan"}:
        return None
    try:
        return float(text)
    except ValueError:
        return None


def numeric_series(rows: list[dict[str, Any]], x_key: str, y_key: str) -> tuple[list[float], list[float]]:
    xs: list[float] = []
    ys: list[float] = []
    for row in rows:
        x = to_float(row.get(x_key))
        y = to_float(row.get(y_key))
        if x is not None and y is not None:
            xs.append(x)
            ys.append(y)
    return xs, ys


def find_finetune_run_dir() -> Path | None:
    candidates = [
        FINETUNE_RUNS_DIR / "demo2_cloud_vit_head_finetune_tpu_resume_first",
        FINETUNE_RUNS_DIR / "from_gcs" / "demo2_cloud_imagenette320_train64_tpu_resume_from_gcs",
        FINETUNE_RUNS_DIR / "demo2_cloud_vit_head_finetune_tpu_time",
    ]
    for candidate in candidates:
        if (candidate / "summary.json").exists() or (candidate / "metrics.csv").exists():
            return candidate
    if FINETUNE_RUNS_DIR.exists():
        for candidate in sorted(FINETUNE_RUNS_DIR.rglob("metrics.csv")):
            return candidate.parent
    return None


def show_summary_fields(summary: dict[str, Any], run_dir: Path) -> None:
    fields = [
        "backend",
        "resumed_from_checkpoint",
        "start_step",
        "final_step",
        "latest_checkpoint_step",
        "trainable_scope",
        "frozen_scope",
        "batch_size",
        "initial_loss",
        "final_loss",
        "mean_step_time_sec",
        "examples_per_second",
        "num_train_classes",
        "num_eval_classes",
    ]
    rows = [f"| Field | Value |", "| --- | --- |"]
    for field in fields:
        if field in summary:
            rows.append(f"| `{field}` | `{summary[field]}` |")
    display(Markdown(f"### Fine-tuning summary: `{run_dir.relative_to(REPO_ROOT)}`\n\n" + "\n".join(rows)))



def build_batch_real_sizes(num_examples: int, batch_size: int) -> list[int]:
    """Return the number of real examples in each padded training batch."""
    if num_examples <= 0 or batch_size <= 0:
        return []
    full_batches, remainder = divmod(num_examples, batch_size)
    sizes = [batch_size] * full_batches
    if remainder:
        sizes.append(remainder)
    return sizes


def median(values: list[float]) -> float | None:
    if not values:
        return None
    ordered = sorted(values)
    midpoint = len(ordered) // 2
    if len(ordered) % 2:
        return ordered[midpoint]
    return (ordered[midpoint - 1] + ordered[midpoint]) / 2


def show_throughput_band_explanation(summary: dict[str, Any] | None, metrics: list[dict[str, Any]]) -> None:
    if not summary:
        return
    train_examples = to_float(summary.get("train_examples"))
    batch_size = to_float(summary.get("batch_size"))
    if train_examples is None or batch_size is None:
        return

    num_examples = int(train_examples)
    batch = int(batch_size)
    real_sizes = build_batch_real_sizes(num_examples, batch)
    if not real_sizes:
        return

    full_batch_eps: list[float] = []
    partial_batch_eps: list[float] = []
    for row in metrics:
        step = to_float(row.get("step"))
        eps = to_float(row.get("examples_per_second"))
        if step is None or eps is None:
            continue
        # The metrics row stores the incremented step. The batch used for that row
        # was selected with the previous state.step.
        batch_index = (int(step) - 1) % len(real_sizes)
        real_size = real_sizes[batch_index]
        if real_size == batch:
            full_batch_eps.append(eps)
        else:
            partial_batch_eps.append(eps)

    pattern = ", ".join(str(size) for size in real_sizes)
    expected_ratio = min(real_sizes) / batch if batch else None
    rows = [
        "| Item | Value |",
        "| --- | --- |",
        f"| train examples | `{num_examples}` |",
        f"| batch size | `{batch}` |",
        f"| real examples per batch | `{pattern}` |",
    ]
    if expected_ratio is not None and min(real_sizes) < batch:
        rows.append(f"| expected partial/full throughput ratio | `{expected_ratio:.3g}` |")
    full_median = median(full_batch_eps)
    partial_median = median(partial_batch_eps)
    if full_median is not None:
        rows.append(f"| median full-batch examples/sec in loaded metrics | `{full_median:.1f}` |")
    if partial_median is not None:
        rows.append(f"| median partial-batch examples/sec in loaded metrics | `{partial_median:.1f}` |")
    if full_median and partial_median:
        rows.append(f"| observed partial/full ratio | `{partial_median / full_median:.3g}` |")

    note = (
        "### Why the examples/sec plot can have two bands\n\n"
        + "\n".join(rows)
        + "\n\n"
        "The partial batch is padded to the fixed JAX shape, but `examples_per_second` "
        "uses only the number of real examples. A low band near one quarter of the "
        "full-batch throughput is expected for a 50-example manifest with batch size 8."
    )
    display(Markdown(note))

def plot_finetune_metrics(run_dir: Path) -> None:
    if plt is None:
        print("matplotlib is unavailable; skipping fine-tuning plots")
        return
    metrics = load_csv_records(run_dir / "metrics.csv")
    eval_metrics = load_csv_records(run_dir / "eval_metrics.csv")
    if not metrics:
        display(Markdown(f"> Missing or empty metrics CSV: `{(run_dir / 'metrics.csv').relative_to(REPO_ROOT)}`"))
        return

    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes_flat = list(axes.flat)
    plot_specs = [
        (metrics, "step", "loss", "training loss vs step", "loss"),
        (metrics, "step", "accuracy", "training accuracy vs step", "accuracy"),
        (eval_metrics, "step", "eval_loss", "eval loss vs step", "eval loss"),
        (eval_metrics, "step", "eval_accuracy", "eval accuracy vs step", "eval accuracy"),
        (metrics, "step", "examples_per_second", "examples/sec vs step", "examples/sec"),
        (metrics, "step", "step_time_sec", "step time vs step", "step time (s)"),
    ]
    for axis, (rows, x_key, y_key, title, ylabel) in zip(axes_flat, plot_specs, strict=True):
        xs, ys = numeric_series(rows, x_key, y_key)
        if xs and ys:
            axis.plot(xs, ys, marker=".", linewidth=1)
            axis.set_title(title)
            axis.set_xlabel("step")
            axis.set_ylabel(ylabel)
            axis.grid(True, alpha=0.3)
        else:
            axis.axis("off")
            axis.text(0.5, 0.5, f"No `{y_key}` data", ha="center", va="center")
    plt.tight_layout()
    plt.show()


run_dir = find_finetune_run_dir()
if run_dir is None:
    display(Markdown("> No local fine-tuning artifacts found. Expected generated files under `runs/vit-finetune/<RUN_NAME>/`."))
else:
    summary = load_json_if_exists(run_dir / "summary.json")
    if summary:
        show_summary_fields(summary, run_dir)
    train_log = run_dir / "train.log"
    if train_log.exists():
        line_count = len(train_log.read_text(encoding="utf-8", errors="replace").splitlines())
        print(f"train.log lines: {line_count}")
    else:
        print(f"missing train.log: {train_log.relative_to(REPO_ROOT)}")
    metrics = load_csv_records(run_dir / "metrics.csv")
    show_throughput_band_explanation(summary, metrics if metrics else [])
    plot_finetune_metrics(run_dir)


## 11. Discussion

### From the pre-TPU plan to current evidence

The pre-TPU report already had the main measurement framework: JAX/Flax inference, manifest batching, final-batch padding, warmup, synchronized timing, JSON artifacts, and Markdown result tables. What changed after the TPU work is that the cloud part is no longer only a plan. There are now retrieved TPU artifacts, TPU Imagenette timing tables, and checkpoint/resume evidence for the optional training extension.

### Inference timing

The TPU Imagenette result tables show a clear batch-size effect. For `val64`, throughput increases from about 1814 images/sec at batch size 1 to about 4337 images/sec at batch size 8. The same pattern also appears in the full validation manifest.

For `val64` and `val256`, where CPU rows exist, the local and external CPU results are much slower than the TPU rows. The full-validation rows should be read as TPU-only timing evidence because no full-validation CPU baseline is present. This is useful evidence for the class presentation, but the comparison is still limited. The machines are different, the benchmark loop is short, and the result is inference-only. It should be presented as a project-specific timing comparison, not as a universal TPU benchmark.

The five-image public-example comparison is the smallest evidence item. It is mainly valuable because it proves the end-to-end workflow: local run, TPU run, artifact retrieval, comparison script, and cleanup. The Imagenette tables are stronger timing evidence because they use larger manifests and multiple batch sizes.

### Visual predictions

The visual grids help check that the JSON artifacts still connect back to image files and per-image predictions. For Imagenette, the notebook intentionally avoids displaying only the first sorted records. This matters because a limit-based manifest can be class-skewed, especially for `val64`. A visually diverse grid should use diverse sampling from the available artifact, or use the full validation artifact when available.

### Fine-tuning and checkpoint resume

The fine-tuning extension is intentionally small. It trains only the classifier head and keeps the ViT backbone frozen. This makes the checkpoint payload small and the resume logic easier to inspect.

The strongest fine-tuning evidence is the GCS restore/resume result. The workflow saved Orbax checkpoints, copied completed checkpoints to GCS, restored the checkpoint on a replacement TPU VM, and continued training from the restored step. That is a meaningful workflow result for a numerical computing course because it connects JAX execution with cloud resource instability and recovery.

The training metrics need one extra caution. If the run uses `manifest_train_balanced_50.txt` with batch size 8, the batches have real sizes `8, 8, 8, 8, 8, 8, 2`. The final batch is padded for a fixed JAX shape, but the reported `examples_per_second` uses only the real examples in the batch. This explains the high and low throughput bands in the metrics plot. A low band around one quarter of the full-batch band is expected and should not be explained as TPU slowdown, checkpoint overhead, or evaluation overhead.

The checkpoint/resume design follows the same principle as the inference benchmark design: make the backend, state, timing, artifacts, and limitations explicit. The report should therefore emphasize the workflow and evidence boundary, not only the final numeric result.


## 12. Conclusion

Demo 2 now gives a complete course-report workflow: local JAX execution, Cloud TPU inference, artifact retrieval, result-table generation, visual sanity checks, and a small checkpoint/resume training extension.

The main lesson is that running JAX on TPU is not just a backend switch. The workflow also needs Google Cloud setup, TPU quota or funding, network and subnet checks, a TPU-compatible JAX installation, backend verification, artifact retrieval, durable checkpoint handling, and cleanup.

The current evidence supports these claims:

- the pretrained ViT inference script runs on local CPU and Cloud TPU;
- TPU inference artifacts were retrieved and summarized in report tables;
- Imagenette 320 timing artifacts show clear batch-size effects on TPU;
- visual grids can inspect saved per-image predictions when local images are available;
- classifier-head training can save Orbax checkpoints, copy completed checkpoints to GCS, and resume from a restored checkpoint on a later TPU VM;
- the observed two-band `examples_per_second` pattern in the fine-tuning metrics is explained by padded-batch accounting, not by TPU slowdown.

The limits are also clear. The inference tables are timing evidence, not accuracy evaluation. The public-example smoke run is too small to be a benchmark study. The classifier-head extension is not full ViT fine-tuning. The GCS checkpoint copy workflow demonstrates recovery for this project, but it is not a complete production training system.

Overall, the project has moved from a local-only pretrained-model demo to a Cloud TPU workflow with reproducible artifacts and a clear report story. That is the main result I would present in class.
